In [11]:
import cloudscraper
import pandas as pd
import time
from datetime import datetime

# URL base sem os filtros manuais
base_url = "https://servicespub.prod.api.aws.grupokabum.com.br/pc-builder/v2/setup/c34cd563-ab09-4587-9784-f6305ce6d960/choices/1"

scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Origin": "https://www.kabum.com.br",
    "Referer": "https://www.kabum.com.br/monte-seu-pc"
}

dados_totais = []
pagina = 1

print("🚀 Iniciando coleta total (Intel + AMD) na KaBuM!...")

while True:
    # Parâmetros ajustados: removemos o 'chips' fixo para trazer tudo
    params = {
        "page_number": pagina,
        "page_size": 50, # Tentamos 50 para reduzir o número de requisições
        "sort": "most_searched"
    }
    
    try:
        response = scraper.get(base_url, headers=headers, params=params)
        
        if response.status_code != 200:
            print(f"Finalizado ou erro. Status: {response.status_code}")
            break
            
        res = response.json()
        produtos = res.get("products", [])
        
        if not produtos:
            print("Nenhum produto a mais encontrado.")
            break
            
        print(f"📦 Coletando página {pagina} ({len(produtos)} itens encontrados)...")
        
        for item in produtos:
            # Extração limpa conforme sua necessidade
            dados_totais.append({
                "nome": item.get("title"),
                "sku": item.get("id"),
                "marca": item.get("manufacturer", {}).get("name"),
                "preco_pix": item.get("price_with_discount"),
                "disponivel": item.get("available"),
                "loja": "KaBuM!",
                "data_coleta": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })
            
        # Se a quantidade de produtos retornada for menor que o page_size, chegamos ao fim
        if len(produtos) < 50:
            break
            
        pagina += 1
        time.sleep(1.5) # Pausa de segurança para evitar bloqueio 403 por excesso de requisições
        
    except Exception as e:
        print(f"Erro durante a execução: {e}")
        break

# Gerando o CSV final
if dados_totais:
    df_completo = pd.DataFrame(dados_totais)
    # Opcional: remover duplicados se houver sobreposição de páginas
    df_completo = df_completo.drop_duplicates(subset=['sku'])
    
    df_completo.to_csv("kabum_todos_processadores.csv", index=False, encoding="utf-8-sig")
    print(f"\n✅ Sucesso! Total de {len(df_completo)} processadores (Intel e AMD) salvos.")
else:
    print("Nenhum dado foi coletado.")

🚀 Iniciando coleta total (Intel + AMD) na KaBuM!...
📦 Coletando página 1 (50 itens encontrados)...
📦 Coletando página 2 (7 itens encontrados)...

✅ Sucesso! Total de 57 processadores (Intel e AMD) salvos.


In [12]:
import cloudscraper
import pandas as pd
from datetime import datetime

# URL da API
URL = "https://www.pichau.com.br/api/request/monte-processador"

# Criação de um objeto Scraper para forçar uma versão específica do navegador
# Isso ajuda a bater o fingerprint do Cloudflare
scraper = cloudscraper.create_scraper(
    browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    }
)

# Criando um Header semelhante a de um navegador real
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://www.pichau.com.br/monte-seu-pc/processador",
    "sec-ch-ua": '"Not A(Brand";v="99", "Google Chrome";v="121", "Chromium";v="121"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
}

try:
    # Tentar acessar a página da Pichau primeiro com o Scraper, para obter cookies de acesso válidos.
    scraper.get("https://www.pichau.com.br", headers=headers)
    
    # Agora tentamos o mesmo procedimento com a URL da API, e armazenamos em "response".
    response = scraper.get(URL, headers=headers)

    # Se houver o status 200 do scraper realizado no "response", verificar:
    if response.status_code == 200:
        res = response.json()
        items = res.get("products", {}).get("items", [])
        
        dados = []
        for cpu in items:
            dados.append({
                "nome": cpu.get("name"),
                "preco_pix": cpu.get("pichau_prices", {}).get("avista"),
                "socket": cpu.get("socket"),
                "loja": "Pichau",
                "data_coleta": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })
            
        df = pd.DataFrame(dados)
        df.to_csv("pichau_processadores.csv", index=False, encoding="utf-8-sig")
        print(f"Sucesso! {len(df)} produtos salvos.")
    else:
        print(f"Ainda bloqueado. Status: {response.status_code}")
        print("A Pichau detectou o script. Recomendado usar Playwright.")

except Exception as e:
    print(f"Erro: {e}")

Sucesso! 36 produtos salvos.
